# Distribuição Territorial de Oportunidades

In [ ]:
YEARS       = range(2014, 2024)

# Backend

In [ ]:
# ── standard-library imports ─────────────────────────────────────────
import os
from pathlib import Path
import re
from typing import Optional, Tuple

# ── third-party imports ──────────────────────────────────────────────
from dotenv import load_dotenv
import geobr
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
from shapely.geometry import LineString
import seaborn as sns
from tobler.util import h3fy


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
plt.style.use('Solarize_Light2')

In [ ]:
pd.options.display.float_format = '{:,.2f}'.format

In [ ]:
DATABASE = os.environ.get('DB_FOLDER')
DATABASE = Path(DATABASE)

OUT_FOLDER = os.environ.get('OUT_FOLDER')
OUT_DIR = Path(OUT_FOLDER) / 'A/oportunidades/rais'
OUT_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv()
GCLOUD_ID = os.getenv("GCLOUD_ID")

# Backup

In [ ]:
inpath = OUT_DIR / f'rais.parquet'
rais = gpd.read_parquet(inpath)

In [ ]:
rais.sample(5)

In [ ]:
rais['cnae_2'] = np.where(
    rais.grupo_cnae == 'P',
    85,
    rais.cnae_2
)

rais['cnae_2'] = np.where(
    rais.grupo_cnae == 'Q',
    86,
    rais.cnae_2
)

In [ ]:
bh = geobr.read_municipality(code_muni=3106200).to_crs(31983).buffer(500)
hexes = pd.concat([
    h3fy(bh, resolution=r).assign(aperture=r).set_index('aperture', append=True)
    for r in range(9, 12)
    ])

In [ ]:
# ------------------------------------------------------
# CNAE → {sector, subtype} mapper (with Transport & Construction)
# ------------------------------------------------------

# Canonical labels
AGRICULTURAL = "agricultural"
INDUSTRIAL = "manufacturing_industrial"
TRANSPORT = "transportation"
CONSTRUCTION = "construction"

CONS_PRIMARY = "consumer_services_primary"
CONS_SECOND = "consumer_services_secondary"

BUS_PRIMARY = "business_services_primary"
BUS_SECOND = "business_services_secondary"


# 1) Division-level defaults (2-digit CNAE divisions)
DIVISION_TO_BUCKET = {
    # A — Agricultura, pecuária, produção florestal, pesca e aquicultura (01–03)
    **{d: (AGRICULTURAL, None) for d in range(1, 4)},

    # B–E — base industrial e utilidades
    **{d: (INDUSTRIAL, None) for d in range(5, 10)},   # B (05–09) Mineração
    **{d: (INDUSTRIAL, None) for d in range(10, 34)},  # C (10–33) Indústrias de transformação
    35: (INDUSTRIAL, None),                             # D (35) Eletricidade e gás
    **{d: (INDUSTRIAL, None) for d in range(36, 40)},  # E (36–39) Água/esgoto/resíduos

    # F — Construção (41–43) → nova categoria própria
    **{d: (CONSTRUCTION, None) for d in range(41, 44)},

    # G — Comércio (45–47)
    45: (CONS_SECOND, None),   # veículos e reparação → consumo, apoio (secundário)
    46: (CONS_SECOND, None),   # atacado TODO: should be B2B (primário)?
    47: (CONS_SECOND, None),   # varejo → consumo; exceções abaixo para ancoragens

    # H — Transporte, armazenagem e correio (49–53)
    **{d: (TRANSPORT, None) for d in range(49, 54)},

    # I — Alojamento e alimentação (55–56)
    55: (CONS_SECOND, None),  # hotéis/alojamento
    56: (CONS_SECOND, None),   # restaurantes/bares

    # J — Informação e comunicação (58–63): B2B (primário)
    **{d: (BUS_PRIMARY, None) for d in range(58, 64)},

    # K — Financeiras e seguros (64–66) → B2B primário
    **{d: (BUS_PRIMARY, None) for d in range(64, 67)},

    # L — Imobiliárias (68) → B2B primário
    68: (BUS_PRIMARY, None),

    # M — Profissionais, científicas e técnicas (69–75) → B2B primário (exceção veterinária)
    **{d: (BUS_PRIMARY, None) for d in range(69, 76)},

    # N — Administrativos e complementares (77–82) → B2B secundário (apoio)
    **{d: (BUS_SECOND, None) for d in range(77, 83)},

    # O — Administração pública (84) → B2B primário (institucional)
    84: (BUS_PRIMARY, None),

    # P — Educação (85) → consumo secundário
    85: (CONS_SECOND, None),

    # Q — Saúde e assistência social (86–88) → consumo primário
    **{d: (CONS_PRIMARY, None) for d in range(86, 89)},

    # R — Artes, cultura, esporte e recreação (90–93) → consumo secundário
    **{d: (CONS_SECOND, None) for d in range(90, 94)},

    # S — Outras atividades de serviços (94–96) → em geral secundário
    94: (BUS_PRIMARY, None),
    **{d: (CONS_SECOND, None) for d in range(95, 97)},

    # T — Serviços domésticos (97) → consumo secundário
    97: (CONS_SECOND, None),

    # U — Organismos internacionais (99) → B2B primário
    99: (BUS_PRIMARY, None),
}

# 2) Group-level (3-digit) overrides for finer distinctions
GROUP_EXCEPTIONS = {
    # M — Veterinária: frequentemente B2C local
    "750": (CONS_SECOND, None),

    # H — Transporte: já é categoria própria; se desejar diferenciar,
    # você pode usar o segundo elemento como subtipo ("passenger"/"freight"):
    "491": (TRANSPORT, "passenger"),
    "492": (TRANSPORT, "passenger"),
    "493": (TRANSPORT, "passenger"),
    "494": (TRANSPORT, "freight"),
    "501": (TRANSPORT, "passenger"),
    "502": (TRANSPORT, "freight"),
    "511": (TRANSPORT, "passenger"),
    "512": (TRANSPORT, "freight"),
    "521": (TRANSPORT, "storage"),
    "522": (TRANSPORT, "auxiliary"),
    "531": (TRANSPORT, "postal"),
    "532": (TRANSPORT, "delivery"),
}

# ---- Helpers ----
def _only_digits(s: str) -> str:
    return re.sub(r"\D", "", s or "")

def _div_from_code(code_digits: str) -> Optional[int]:
    if len(code_digits) >= 2:
        return int(code_digits[:2])
    return None

def _group_from_code(code_digits: str) -> Optional[str]:
    if len(code_digits) >= 3:
        return code_digits[:3]
    return None

def classify_cnae(
    cnae_code: str,
) -> Tuple[str, Optional[str]]:
    """
    Map a CNAE code (division/group/class) to:
      (sector_label, subtype_or_None)

    sector_label ∈ {
        'Agricultural',
        'Manufacturing/Industrial',
        'Transportation',
        'Construction',
        'Consumer Services — primary',
        'Consumer Services — secondary',
        'Business Services — primary',
        'Business Services — secondary'
    }
    Note: Only Consumer/Business Services carry primary/secondary by design.
    """
    digits = _only_digits(str(cnae_code))

    # Group-level overrides
    g = _group_from_code(digits)
    if g and g in GROUP_EXCEPTIONS:
        return GROUP_EXCEPTIONS[g]

    # Division defaults
    d = _div_from_code(digits)
    if d in DIVISION_TO_BUCKET:
        sector, subtype = DIVISION_TO_BUCKET[d]
    else:
        # Conservative fallback
        sector, subtype = (BUS_SECOND, None)

    return (sector, subtype)


def map_cnae_series(
    df: pd.DataFrame,
    cnae_col: str,
    out_col_sector: str = "sector",
    out_col_subtype: str = "sector_subtype",
) -> pd.DataFrame:
    def _row_map(row):
        code = row[cnae_col]
        sector, subtype = classify_cnae(str(code))
        return pd.Series({out_col_sector: sector, out_col_subtype: subtype})
    mapped = df.apply(_row_map, axis=1)
    return pd.concat([df, mapped], axis=1)


In [ ]:
rais = map_cnae_series(
    rais.dropna(subset='cnae_2'),
    cnae_col='cnae_2',
    out_col_sector='sector',
    out_col_subtype='sector_subtype',
)

In [ ]:
print(rais.shape)
rais.sample(5)

In [ ]:
inpath  = DATABASE / 'beaga/ENDERECO.zip'

addresses = (
    gpd.read_file(inpath)
    .dropna(subset=['NUMERO_IMO', 'CEP'])
    .astype({
        'NUMERO_IMO': int,
        'CEP': int,
        })
    .astype({'CEP': str})
    .sort_values(['CEP', 'NUMERO_IMO'])
    .pipe(
        lambda gdf: gdf.loc[
                        (gdf.CEP.map(len) == 8)
                        & gdf.CEP.isin(rais.cep.astype(str).unique())
                        ]
        )
    .pipe(
        lambda gdf: gdf.loc[gdf.geometry.notnull() & gdf.geometry.is_valid]
        )
    # Column names to lowercase
    .pipe(lambda d: d.rename(str.lower, axis="columns"))
)

In [ ]:
def assign_geometry_by_cep(
    establishments: gpd.GeoDataFrame,
    addresses: gpd.GeoDataFrame,
    cep_col: str = "cep",
    geometry_col: str = "geometry",
    seed: int | None = 42,
) -> gpd.GeoDataFrame:
    """
    For each establishment, sample (with replacement) a geometry from an
    address with the same CEP and overwrite the establishment geometry.
    If the CEP is missing or there are no address geometries for that CEP,
    the original establishment geometry is preserved.

    Parameters
    ----------
    establishments : GeoDataFrame
        Input establishments with existing geometries (to be preserved when
        no CEP match is available).
    addresses : GeoDataFrame
        Address points with CEP and geometries to draw from.
    cep_col : str, default "cep"
        Column name holding the CEP in both dataframes.
    geometry_col : str, default "geometry"
        Geometry column name.
    seed : int | None, default 42
        Seed for the random generator (None for non-deterministic draws).

    Returns
    -------
    GeoDataFrame
        Establishments with geometries reassigned by CEP where possible,
        otherwise unchanged.
    """
    rng = np.random.default_rng(seed)

    # Work on copies and align CRS so sampled geoms match establishments
    out = establishments.copy()
    addr = addresses.copy()
    if addr.crs != out.crs:
        addr = addr.to_crs(out.crs)

    # Precompute candidate geometries per CEP (skip missing geometries)
    geom_dict: dict = (
        addr.groupby(cep_col)[geometry_col]
        .apply(lambda s: s.dropna().to_numpy(copy=False))
        .to_dict()
    )

    # Establishments' CEPs
    ceps = out[cep_col]

    # Eligible rows: have CEP and have candidates for that CEP
    eligible_mask = ceps.notna() & ceps.map(
        lambda c: len(geom_dict.get(c, ())) > 0
    )

    if eligible_mask.any():
        # Group only the eligible subset by CEP to sample efficiently
        eligible_ceps = ceps[eligible_mask]
        for cep, idx in eligible_ceps.groupby(eligible_ceps).groups.items():
            candidates = geom_dict[cep]
            n = len(idx)
            picks = rng.integers(0, len(candidates), size=n)
            # Overwrite only eligible rows; others keep original geometry
            out.loc[idx, geometry_col] = list(candidates[picks])

    # Ensure a valid GeoDataFrame with correct geometry/CRS
    out = gpd.GeoDataFrame(out, geometry=geometry_col, crs=out.crs)
    return out


In [ ]:
rais = assign_geometry_by_cep(
    establishments=rais,
    addresses=addresses,
    cep_col="cep",
    geometry_col="geometry",
)

rais.shape

In [ ]:
outpath = OUT_DIR / f'rais_treated.parquet'
rais = rais.astype({'cnae_2': str}).to_parquet(outpath)

In [ ]:
def aggregate_rais_by_hex(
    rais: gpd.GeoDataFrame,
    hexes: gpd.GeoDataFrame,
    year_col: str = "ano",
    sector_col: str = "sector",
    weight_col: str = "quantidade_vinculos_ativos",
    hex_id_col: str = "hex_id",
    aperture_col: str = "aperture",
) -> tuple[gpd.GeoDataFrame, gpd.GeoDataFrame]:
    """
    Aggregate RAIS by hex and year (sum and count) and retain geometry.

    Returns two GeoDataFrames (sum, count) indexed by (hex_id, aperture, year)
    with one geometry per (hex_id, aperture) replicated across years. Empty
    hex–years are explicit with zeros.
    """
    if rais.crs != hexes.crs:
        rais = rais.to_crs(hexes.crs)

    # Spatial assignment: point → hex (covered_by counts boundary points)
    joined = gpd.sjoin(
        rais[[year_col, sector_col, weight_col, "geometry"]],
        hexes[[hex_id_col, aperture_col, "geometry"]],
        how="inner",
        predicate="covered_by",
    )[[year_col, sector_col, weight_col, hex_id_col, aperture_col]]

    # If a point matches multiple hexes on borders, keep one deterministically
    joined = (
        joined.reset_index(names="row_id")
        .sort_values(["row_id", aperture_col, hex_id_col])
        .drop_duplicates(subset=["row_id", aperture_col], keep="first")
        .set_index("row_id")
    )

    # Helpers -----------------------------------------------------------------
    def _pivot_sum(df: pd.DataFrame) -> pd.DataFrame:
        tmp = (
            df.groupby(
                [hex_id_col, aperture_col, year_col, sector_col], dropna=False
            )[weight_col]
            .sum()
            .rename("value")
            .reset_index()
        )
        out = (
            tmp.pivot_table(
                index=[hex_id_col, aperture_col, year_col],
                columns=sector_col,
                values="value",
                fill_value=0,
                observed=False,
            )
            .rename_axis(None, axis=1)
            .reset_index()
        )
        return out

    def _pivot_cnt(df: pd.DataFrame) -> pd.DataFrame:
        tmp = (
            df.groupby(
                [hex_id_col, aperture_col, year_col, sector_col], dropna=False
            )
            .size()
            .rename("value")
            .reset_index()
        )
        out = (
            tmp.pivot_table(
                index=[hex_id_col, aperture_col, year_col],
                columns=sector_col,
                values="value",
                fill_value=0,
                observed=False,
            )
            .rename_axis(None, axis=1)
            .reset_index()
        )
        return out

    gdf_sum_wide = _pivot_sum(joined)
    gdf_cnt_wide = _pivot_cnt(joined)

    # Scaffold: unique (hex_id, aperture, geometry) × unique years ------------
    keys = hexes[[hex_id_col, aperture_col, "geometry"]].drop_duplicates(
        subset=[hex_id_col, aperture_col]
    )
    years = rais[[year_col]].drop_duplicates().sort_values(year_col)
    scaffold = keys.merge(years, how="cross")
    scaffold = gpd.GeoDataFrame(scaffold, geometry="geometry", crs=hexes.crs)

    # Attach aggregates to scaffold; keep explicit zero rows per year ----------
    gdf_sum = scaffold.merge(
        gdf_sum_wide, on=[hex_id_col, aperture_col, year_col], how="left"
    )
    gdf_cnt = scaffold.merge(
        gdf_cnt_wide, on=[hex_id_col, aperture_col, year_col], how="left"
    )

    # Fill only measure columns with zeros
    for df in (gdf_sum, gdf_cnt):
        val_cols = [
            c for c in df.columns
            if c not in {hex_id_col, aperture_col, year_col, "geometry"}
        ]
        df[val_cols] = df[val_cols].fillna(0)

    return gdf_sum, gdf_cnt


In [ ]:
gdf_sum, gdf_cnt = aggregate_rais_by_hex(rais, hexes.reset_index())

In [ ]:
def check_mass_preservation(
    rais: gpd.GeoDataFrame,
    hex_sum: gpd.GeoDataFrame,
    hex_cnt: gpd.GeoDataFrame,
    year_col: str = "ano",
    weight_col: str = "quantidade_vinculos_ativos",
    hex_id_col: str = "hex_id",
    aperture_col: str = "aperture",
    hexes_for_coverage: gpd.GeoDataFrame | None = None,
    abs_tol: float = 1e-6,
    rel_tol: float = 1e-6,
) -> pd.DataFrame:
    """
    Verify 'mass' preservation per (aperture, year) for:
      1) sum of vínculos (weights) and
      2) establishment counts.

    If `hexes_for_coverage` is provided, RAIS totals are computed only for
    establishments that fall within any hex polygon (coverage-matched).
    """

    # --- RAIS totals by year (reference)
    if hexes_for_coverage is not None:
        # restrict to points that intersect any hex (fair coverage)
        if rais.crs != hexes_for_coverage.crs:
            rais = rais.to_crs(hexes_for_coverage.crs)
        covered_idx = gpd.sjoin(
            rais[[year_col, weight_col, "geometry"]],
            hexes_for_coverage[["geometry"]],
            how="inner",
            predicate="within",
        ).index.unique()
        rais_ref = rais.loc[covered_idx]
    else:
        rais_ref = rais

    rais_totals = (
        rais_ref.groupby(year_col, dropna=False)
        .agg(
            rais_weight_total=(weight_col, "sum"),
            rais_count_total=(year_col, "size"),
        )
        .reset_index()
    )

    # --- helper: sector columns = numeric and not keys/geometry
    def _sector_cols(df: pd.DataFrame) -> list[str]:
        exclude = {hex_id_col, aperture_col, year_col, "geometry"}
        return [
            c for c in df.columns
            if c not in exclude and pd.api.types.is_numeric_dtype(df[c])
        ]

    # --- totals from hex panels (sum across sectors, then across hexes)
    def _panel_totals(panel: gpd.GeoDataFrame, out_name: str) -> pd.DataFrame:
        df = panel.copy()
        sectors = _sector_cols(df)
        by_ay = df.groupby([aperture_col, year_col], dropna=False)[sectors] \
                  .sum()
        totals = by_ay.sum(axis=1).rename(out_name).reset_index()
        return totals

    sum_totals = _panel_totals(hex_sum, "hex_weight_total")
    cnt_totals = _panel_totals(hex_cnt, "hex_count_total")

    # --- assemble report
    report = sum_totals.merge(
        cnt_totals, on=[aperture_col, year_col], how="outer"
    ).merge(rais_totals, on=year_col, how="left")

    for c in ("hex_weight_total", "hex_count_total",
              "rais_weight_total", "rais_count_total"):
        if c in report:
            report[c] = report[c].fillna(0)

    # diffs and relative diffs
    report["diff_weight"] = report["hex_weight_total"] - \
        report["rais_weight_total"]
    report["diff_count"] = report["hex_count_total"] - \
        report["rais_count_total"]

    def _reldiff(diff: pd.Series, denom: pd.Series) -> pd.Series:
        safe = denom.where(denom != 0, 1.0)
        rel = diff.abs() / safe
        return rel.where(~((denom == 0) & (diff == 0)), 0.0)

    report["rel_diff_weight"] = _reldiff(
        report["diff_weight"], report["rais_weight_total"]
    )
    report["rel_diff_count"] = _reldiff(
        report["diff_count"], report["rais_count_total"]
    )

    report["ok_weight"] = (
        (report["diff_weight"].abs() <= abs_tol)
        | (report["rel_diff_weight"] <= rel_tol)
    )
    report["ok_count"] = (
        (report["diff_count"].abs() <= abs_tol)
        | (report["rel_diff_count"] <= rel_tol)
    )

    cols = [
        aperture_col, year_col,
        "hex_weight_total", "rais_weight_total", "diff_weight",
        "rel_diff_weight", "ok_weight",
        "hex_count_total", "rais_count_total", "diff_count",
        "rel_diff_count", "ok_count",
    ]
    return report[cols].sort_values([aperture_col, year_col]).reset_index(
        drop=True
    )


In [ ]:
mass_report = check_mass_preservation(rais, gdf_sum, gdf_cnt)

mass_report

In [ ]:
h = hexes.xs(key=9, level='aperture')

r = rais.loc[rais.ano == 2019]

In [ ]:
db = pd.read_csv('aop_landuse_2019_v2.csv')
db = db.loc[(db.name_muni == 'Belo Horizonte'), ['id_hex', 'T001']]

In [ ]:
(
    hexes.xs(key=9, level='aperture').merge(db, left_index=True, right_on='id_hex')
    .plot('T001', scheme='naturalbreaks', cmap='inferno')
    )

plt.show()

In [ ]:
gdf_sum['T001'] = gdf_sum.iloc[:, 4:].sum(1)

In [ ]:
(
    gdf_sum
    .loc[
        (gdf_sum.aperture == 9)
        & (gdf_sum.ano == 2023)
          ]
    .fillna({'T001': 0})
    .plot('T001', scheme='naturalbreaks', cmap='inferno')
    )
plt.show()

In [ ]:
outpath = OUT_DIR / 'jobs_by_hex.parquet'
gdf_sum.to_parquet(outpath)

outpath = OUT_DIR / 'establishments_by_hex.parquet'
gdf_cnt.to_parquet(outpath)


- se o modelo vai considerar alocar populacao, considerar alocar vagas de emprego
- cogitar pensar usar os dados de vinculos para fazer alguma desagregacao por rendimento: se tomados os 10%mais ricos, alocados primeiro, qual a localiza'cao preferencial deles?
    - talvez em vez de low/high density, pensar em low/high income
    - ainda, pensar em separar domicilios/habitantes por quintis de renda ou similar
    - relacao entre rendimento nomimal/medio/per capita e renda do responsavel
- em vez de ibeu, pegar variaveis de ambiente construido comuns aos dois censos e usar individualmente ou fazer algum ibeu a parte
- procurar dados de seguranca publica em BH (perguntar andre)
- check density thresholds in nigraha (by page 212)
- think on how to cap densities and how to use some version of disutilities to do it (see eq. 6-12 in nugraha and white et al/van vliet)
    talvez tirar algo da OD, alguma relacao entre densidade e tempo de viagem

# Exercício

**Objetivo:** Compreender os fundamentos da análise de padrões de pontos (*Point Pattern Analysis*) e explorar técnicas para visualizar e interpretar eventos espaciais representados como pontos georreferenciados.

Abaixo, está um mapa com uma amostra ds emprgos da RAIS, representados por pontos que são a localização geogr;afica dos respectivos CEPs. 

Pontos no espaço podem representar fenômenos distintos dependendo de como os interpretamos. No caso da RAIS, ao geolocalizarmos estabelecimentos com base em seus CEPs, podemos tratá-los como **eventos que poderiam ocorrer em qualquer lugar da cidade, mas que se concentram em alguns pontos específicos**. Nesse contexto, nos interessa entender *por que* esses eventos (estabelecimentos) ocorrem em certas áreas e *como* estão distribuídos espacialmente.

Esse tipo de análise — conhecida como **padrão de pontos** — busca descrever e modelar a distribuição dos eventos no espaço. A distribuição pode ser:
- **Aleatória**, sem nenhuma estrutura evidente;
- **Agrupada**, sugerindo concentração em certas regiões;
- Ou **dispersa**, indicando distanciamento entre eventos.

A análise nos permite transformar listas de coordenadas em **fenômenos espaciais interpretáveis**, ajudando a responder perguntas como:
- Onde estão os centros de atividade econômica?
- Existem “vazios” no espaço urbano sem estabelecimentos?
- Há evidência de *clusters* (aglomerados) por setor ou tipo de empresa?

Trabalhe com os dados geolocalizados da RAIS, tratando cada ponto como uma observação de um processo espacial subjacente. A proposta é aplicar conceitos e ferramentas de análise de padrões pontuais para caracterizar a estrutura espacial dos dados — usando bibliotecas como `geopandas`, `pointpats` e `PySAL`.

Acompanhe o capítulo Point Pattern Analysis do livro [Geographic Data Science with Python](https://geographicdata.science/book/notebooks/08_point_pattern_analysis.html) e reproduza o exercício prático lá contido com os municípios acima. Documente o passo a passo e discuta e analise seus achados.

→ Note que exportamos dados de três anos (2021, 2022 e 2023). Extraia 2023 e faça as análises com esse subconjunto.
→ Você pode trabalhar com os dados agregados, mas pode ser interessante ver como a análise muda para as diferentes categorias do CONCLA.

In [ ]:
OUT_PARQUET    = (
    Path("../outputs/sociodemografia")
    / f"sociodemografia_hex_r9_{'-'.join(str(m) for m in MUNIS)}.parquet"
)
OUT_PARQUET

In [ ]:
hexes = (
    gpd
    .read_parquet(OUT_PARQUET)
    .reindex(
        columns=[
            'year',
            'geometry'
            ]
        )
    .pipe(
        lambda x: x.loc[x.year==2022]
        )
    .reset_index()
)